> ▶️ **Run every cell in order, top to bottom.** If a cell errors, re-run the setup cell first (it is idempotent), then continue.

# Module 02 — Write Problems (Colab)

A real agent files a regulatory obligation into a real Postgres database. Run it twice and it files the same obligation twice — because the model words it differently each time. You'll fix that.

The model's answers are replayed from recordings (fast, 100% reproducible). The facilitator runs the live model on screen; you can too (see the last cell). **Run the cells top to bottom.**

## 1. Get the code and set up the environment

In [ ]:
!git clone https://github.com/sanbhaumik/workshop-designing-data-infra-for-ai-agents.git repo
%cd repo

In [ ]:
# Participant path: Postgres + recorded model outputs (skips the slow model install).
!SKIP_OLLAMA=1 bash setup.sh

In [ ]:
import os
os.environ['NOVA_LLM'] = 'frozen'
os.environ['DATABASE_URL'] = 'postgresql://postgres@localhost:5432/nova'
print('environment configured')

In [ ]:
!python preflight.py

## 2. Watch the agent fail

Two runs (a retry). Watch the two REASON lines — same obligation, different wording — and the regulator receiving **two** filings.

In [ ]:
!python modules/02_write_path/naive.py

See the duplicate rows in the **real database** with SQL:

In [ ]:
!psql "$DATABASE_URL" -c "SELECT client_id, left(obligation_text,60) AS obligation FROM filings;"

## 3. Fix it

The cell below is `your_fix.py`. Right now `obligation_identity` keys on `obligation_text`, which changes every run. **Edit the last line** so the key is derived from the agent's stable intent — `client_id` and `source_doc` — then re-run this cell to save it.

Hint: `return hashlib.sha256(f"{client_id}|{source_doc}".encode("utf-8")).hexdigest()`

In [ ]:
%%writefile modules/02_write_path/your_fix.py
import hashlib


def obligation_identity(client_id: str, source_doc: str, obligation_text: str) -> str:
    """Return a STABLE idempotency key identifying this obligation."""
    # TODO: key on the agent's INTENT (client_id, source_doc), NOT obligation_text.
    return hashlib.sha256(obligation_text.encode("utf-8")).hexdigest()

In [ ]:
!python -m pytest modules/02_write_path/test_write.py -v

## 4. See it land

Before/after: the naive identity files twice; your identity files once.

In [ ]:
!python modules/02_write_path/compare.py

### Optional: run the real local model yourself

Everything above replayed the model's answers from recordings. To run the actual open-source model (slower — installs Ollama and pulls a ~1.3 GB model; a T4 GPU runtime helps), then re-run the cells above:

```python
!bash setup.sh
os.environ['NOVA_LLM'] = 'ollama'
os.environ['OLLAMA_MODEL'] = 'llama3.2:1b'
```